**Stage 2: person removal (inpainting).** YOLOv8x-Seg person masks -> union -> close 25x25 -> dilate 15x15 (elliptical kernels) -> 21x21 Gaussian feather -> blend with the background frame. Output: `<vid>-output_inpainted.mp4` (cover video).

In [ ]:
!pip install ultralytics
import cv2
import numpy as np
import os

import cv2
import numpy as np
import os
from ultralytics import YOLO

# ── Paths ────────────────────────────────────────────────────────────────────
video_path   = '/content/vid10.avi'
bg_img_path  = '/content/vid10-BG_frame.jpg'
output_path  = 'vid10-output_inpainted.mp4'
# ─────────────────────────────────────────────────────────────────────────────

# Load YOLOv8 segmentation model (downloads automatically on first run)
model = YOLO('yolov8x-seg.pt')

cap = cv2.VideoCapture(video_path)
background_image = cv2.imread(bg_img_path)

frame_width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps          = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

resized_bg = cv2.resize(background_image, (frame_width, frame_height))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter(output_path, fourcc, fps, (frame_width, frame_height))

# Morphological kernels for mask cleanup
closing_kernel  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (25, 25))
dilation_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))

# COCO class index for 'person' is 0
PERSON_CLASS_ID = 0

frame_count = 0

print(f"Processing {total_frames} frames...")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Run segmentation on this frame
    # classes=[0] tells YOLO to only detect persons — faster, no false masks
    results = model(frame, classes=[PERSON_CLASS_ID], verbose=False)

    # Build a combined person mask for all detected people in this frame
    combined_mask = np.zeros((frame_height, frame_width), dtype=np.uint8)

    for r in results:
        # r.masks contains segmentation polygon masks (one per detected person)
        if r.masks is not None:
            for mask_data in r.masks.data.cpu().numpy():
                # mask_data is a float32 array, resize to frame dimensions
                person_mask = cv2.resize(mask_data, (frame_width, frame_height))
                person_mask = (person_mask > 0.5).astype(np.uint8) * 255

                # Add this person's mask into the combined mask
                combined_mask = cv2.bitwise_or(combined_mask, person_mask)

        # Fallback: if segmentation masks aren't available, use bounding boxes
        elif r.boxes is not None:
            for box in r.boxes:
                if int(box.cls[0]) == PERSON_CLASS_ID:
                    x1, y1, x2, y2 = map(int, box.xyxy[0])
                    cv2.rectangle(combined_mask, (x1, y1), (x2, y2), 255, -1)

    if combined_mask.max() > 0:
        # Close holes within person silhouettes (gaps between limbs, etc.)
        combined_mask = cv2.morphologyEx(combined_mask, cv2.MORPH_CLOSE, closing_kernel)

        # Expand slightly to catch silhouette edge pixels
        combined_mask = cv2.dilate(combined_mask, dilation_kernel, iterations=1)

        # Feather edges for a natural blend
        feathered = cv2.GaussianBlur(combined_mask, (21, 21), 0)
        alpha = feathered.astype(np.float32) / 255.0
        alpha_3ch = cv2.merge([alpha, alpha, alpha])

        frame_f = frame.astype(np.float32)
        bg_f    = resized_bg.astype(np.float32)

        # Person pixels → background; everything else → original frame
        final_frame = (bg_f * alpha_3ch + frame_f * (1.0 - alpha_3ch)).astype(np.uint8)
    else:
        # No people detected — write original frame untouched
        final_frame = frame

    writer.write(final_frame)
    frame_count += 1

    if frame_count % 25 == 0:
        print(f"  {frame_count}/{total_frames} frames done...")

cap.release()
writer.release()

if os.path.exists(output_path):
    print(f"\nDone. Output saved to '{output_path}'.")
else:
    print("\nError: output file was not created.")

In [ ]:
import IPython.display as ipd

# Display the generated video
ipd.Video('output_inpainted.mp4', embed=True)

In [ ]:
from google.colab import files
files.download('output_inpainted.mp4')